# 🅿️ Notebook 01 : Phase 1 — Scénarios de Simulation & Logique Métier
> **Projet Semestriel : Visual Tracking & Intelligent Applications**  
> **5ème Année Génie Informatique — AU 2026–2027 (Dr. Ing. Rihab LAAJILI)**  
> **Livrable Phase 1 (Cadrage & Validation Métier)**

---

### 📋 Objectif du Notebook :
Démontrer de manière isolée et reproductible les 4 scénarios d'application métier exigés par le cahier des charges :
1. **Scénario 1 : Stationnement Régulier & Conforme** (Insertion fluide, point d'ancrage sol $P_{\text{sol}}$, Dwell Time chronométré).
2. **Scénario 2 : Stationnement à Cheval sur la Ligne** (Incivilité bloquant 2 places contiguës par échantillonnage tripartite des roues).
3. **Scénario 3 : Risque de Collision Évité (Near-Miss)** (Alerte préventive lors du recul et levée d'alerte lors du freinage d'urgence).
4. **Scénario 4 : Collision Réelle Confirmée** (Impact physique direct en allée centrale avec ondes de choc).


In [ ]:
# 1. Installation des dépendances (Google Colab / Kaggle)
!pip install -q ultralytics shapely opencv-python matplotlib pandas

import cv2
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import deque
from IPython.display import HTML
from base64 import b64encode
import os

print('✅ Environnement et bibliothèques initialisés !')


## 1. Définition Géométrique des Places & Voie de Circulation


In [ ]:
PARKING_CONFIG = [
  {"id": 1, "name": "Slot_01", "polygon": [[120, 100], [260, 100], [260, 280], [120, 280]], "max_dwell_minutes": 15},
  {"id": 2, "name": "Slot_02", "polygon": [[280, 100], [420, 100], [420, 280], [280, 280]], "max_dwell_minutes": 15},
  {"id": 3, "name": "Slot_03", "polygon": [[440, 100], [580, 100], [580, 280], [440, 280]], "max_dwell_minutes": 15},
  {"id": 4, "name": "Slot_04", "polygon": [[120, 440], [260, 440], [260, 620], [120, 620]], "max_dwell_minutes": 15},
  {"id": 5, "name": "Slot_05", "polygon": [[280, 440], [420, 440], [420, 620], [280, 620]], "max_dwell_minutes": 15},
  {"id": 6, "name": "Slot_06", "polygon": [[440, 440], [580, 440], [580, 620], [440, 620]], "max_dwell_minutes": 15}
]

os.makedirs('configs', exist_ok=True)
import json
with open('configs/parking_slots.json', 'w') as f:
    json.dump(PARKING_CONFIG, f, indent=2)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.set_facecolor('#222222')
ax.set_xlim(0, 1280)
ax.set_ylim(720, 0)

lane_patch = patches.Rectangle((0, 300), 1280, 120, facecolor='#f39c12', alpha=0.2, label='Voie de Circulation')
ax.add_patch(lane_patch)
ax.plot([0, 1280], [360, 360], color='#ffcc00', linestyle='--', linewidth=2)
ax.plot([140, 140], [300, 420], color='#00ffff', linewidth=3, label='Ligne Virtuelle Entrée')

for slot in PARKING_CONFIG:
    poly = np.array(slot['polygon'])
    patch = patches.Polygon(poly, closed=True, facecolor='#2ecc71', edgecolor='#27ae60', alpha=0.45, linewidth=2)
    ax.add_patch(patch)
    cx, cy = np.mean(poly[:, 0]), np.mean(poly[:, 1])
    ax.text(cx, cy, slot['name'], color='white', weight='bold', fontsize=10, ha='center', va='center')

plt.title('Plan Géométrique du Parking (Baseline Phase 1)', fontsize=13, color='white')
fig.patch.set_facecolor('#181818')
ax.tick_params(colors='white')
plt.legend(facecolor='#333333', edgecolor='none', labelcolor='white', loc='upper right')
plt.tight_layout()
plt.show()


## 2. Moteur Spatial & Filtrage Temporel (Anti-Flickering)


In [ ]:
class SpatialEngine:
    def __init__(self, config_slots, hysteresis_window=30, occupied_threshold=0.8, free_threshold=0.2,
                 collision_dist=75.0, near_miss_dist=120.0):
        self.slots = config_slots
        self.hysteresis_window = hysteresis_window
        self.occupied_threshold = occupied_threshold
        self.free_threshold = free_threshold
        self.collision_dist = collision_dist
        self.near_miss_dist = near_miss_dist

        self.history = {s['id']: deque(maxlen=self.hysteresis_window) for s in self.slots}
        self.occupancy_state = {s['id']: False for s in self.slots}
        self.dwell_data = {
            s['id']: {'car_id': None, 'occupied_since': None, 'elapsed_seconds': 0.0}
            for s in self.slots
        }
        self.tripwire_x = 140
        self.crossed_ids = set()
        self.total_entries = 0
        self.prev_positions = {}
        self.pair_distance_windows = {}

    @staticmethod
    def get_bottom_contact_point(bbox):
        x1, y1, x2, y2 = bbox
        return int((x1 + x2) / 2.0), int(y2)

    def is_point_in_slot(self, pt, slot_polygon):
        pts = np.array(slot_polygon, np.int32)
        return cv2.pointPolygonTest(pts, (float(pt[0]), float(pt[1])), measureDist=False) >= 0

    def compute_overlap(self, b1, b2):
        xA, yA = max(b1[0], b2[0]), max(b1[1], b2[1])
        xB, yB = min(b1[2], b2[2]), min(b1[3], b2[3])
        inter = max(0, xB - xA) * max(0, yB - yA)
        min_area = min((b1[2]-b1[0])*(b1[3]-b1[1]), (b2[2]-b2[0])*(b2[3]-b2[1]))
        return (inter / float(min_area)) if min_area > 0 else 0.0

    def update(self, tracked_vehicles, current_time):
        instant_assignments = {s['id']: None for s in self.slots}
        cars_in_slots = set()
        line_straddle_alerts = []

        # 1. Association spatiale et détection à cheval sur la ligne
        for veh in tracked_vehicles:
            cid = veh['id']
            bbox = veh['bbox']
            px, py = self.get_bottom_contact_point(bbox)
            w = bbox[2] - bbox[0]
            p_left = (int(bbox[0] + w * 0.25), int(bbox[3]))
            p_right = (int(bbox[2] - w * 0.25), int(bbox[3]))

            if px > self.tripwire_x and cid not in self.crossed_ids:
                self.crossed_ids.add(cid)
                self.total_entries += 1

            matched = []
            for slot in self.slots:
                sid = slot['id']
                poly = slot['polygon']
                if self.is_point_in_slot((px, py), poly) or (self.is_point_in_slot(p_left, poly) and self.is_point_in_slot(p_right, poly)):
                    matched.append(sid)

            if len(matched) >= 2:
                line_straddle_alerts.append({'car_id': cid, 'slot_ids': matched, 'bbox': bbox})
                for sid in matched:
                    instant_assignments[sid] = cid
                    cars_in_slots.add(cid)
            elif len(matched) == 1:
                instant_assignments[matched[0]] = cid
                cars_in_slots.add(cid)

        # 2. Hystérésis & Dwell time
        for slot in self.slots:
            sid = slot['id']
            cid = instant_assignments[sid]
            self.history[sid].append(cid is not None)
            ratio = sum(self.history[sid]) / len(self.history[sid])
            prev = self.occupancy_state[sid]

            if ratio >= self.occupied_threshold:
                new_state = True
            elif ratio <= self.free_threshold:
                new_state = False
            else:
                new_state = prev

            self.occupancy_state[sid] = new_state
            dwell = self.dwell_data[sid]
            if new_state:
                if not prev or dwell['occupied_since'] is None:
                    dwell['occupied_since'] = current_time
                    dwell['car_id'] = cid
                dwell['elapsed_seconds'] = current_time - dwell['occupied_since']
                if cid is not None:
                    dwell['car_id'] = cid
            else:
                dwell['occupied_since'] = None
                dwell['car_id'] = None
                dwell['elapsed_seconds'] = 0.0

        # 3. Collisions & Near-Miss Fenêtrées (Convergence réelle obligatoire)
        collision_alerts = []
        n_veh = len(tracked_vehicles)
        for i in range(n_veh):
            for j in range(i + 1, n_veh):
                v1, v2 = tracked_vehicles[i], tracked_vehicles[j]
                c1, c2 = v1['id'], v2['id']
                if c1 in cars_in_slots and c2 in cars_in_slots:
                    continue

                b1, b2 = v1['bbox'], v2['bbox']
                p1 = ((b1[0]+b1[2])/2.0, (b1[1]+b1[3])/2.0)
                p2 = ((b2[0]+b2[2])/2.0, (b2[1]+b2[3])/2.0)
                d_c = np.hypot(p1[0]-p2[0], p1[1]-p2[1])

                pair = tuple(sorted((c1, c2)))
                if pair not in self.pair_distance_windows:
                    self.pair_distance_windows[pair] = deque(maxlen=15)
                self.pair_distance_windows[pair].append((current_time, d_c))

                win = self.pair_distance_windows[pair]
                ov = self.compute_overlap(b1, b2)

                if len(win) >= 6:
                    t_old, d_old = win[0]
                    dt = max(1e-3, current_time - t_old)
                    delta_d = d_c - d_old
                    closing_spd = delta_d / dt
                    if (ov > 0.20 or d_c < self.collision_dist) and (delta_d < -20.0 and closing_spd < -50.0):
                        collision_alerts.append({
                            'type': 'COLLISION', 'car1_id': c1, 'car2_id': c2,
                            'dist': d_c, 'closing_speed': abs(closing_spd),
                            'pos': (int((p1[0]+p2[0])/2), int((p1[1]+p2[1])/2))
                        })
                    elif d_c < self.near_miss_dist and (delta_d < -20.0 and closing_spd < -50.0):
                        collision_alerts.append({
                            'type': 'NEAR_MISS', 'car1_id': c1, 'car2_id': c2,
                            'dist': d_c, 'closing_speed': abs(closing_spd),
                            'pos': (int((p1[0]+p2[0])/2), int((p1[1]+p2[1])/2))
                        })

        tot = len(self.slots)
        occ = sum(1 for v in self.occupancy_state.values() if v)
        return {
            'total_slots': tot,
            'occupied_count': occ,
            'free_count': tot - occ,
            'occupancy_rate': (occ / tot * 100.0) if tot > 0 else 0.0,
            'states': self.occupancy_state,
            'dwell_data': self.dwell_data,
            'line_straddle_alerts': line_straddle_alerts,
            'collision_alerts': collision_alerts,
            'total_entries': self.total_entries
        }

print('[OK] SpatialEngine Phase 1 configuré !')


## 3. Visualiseur Graphique & Lecteur Vidéo HTML5


In [ ]:
class ParkingVisualizer:
    def __init__(self, slot_defs):
        self.slot_defs = slot_defs

    def render(self, frame, analytics, vehicles, fps=30.0):
        vis = frame.copy()
        overlay = frame.copy()
        h, w = vis.shape[:2]

        states = analytics.get('states', {})
        dwell = analytics.get('dwell_data', {})
        straddles = analytics.get('line_straddle_alerts', [])
        collisions = analytics.get('collision_alerts', [])
        entries = analytics.get('total_entries', 0)

        # Rendu des places
        straddled_slots = set()
        for stra in straddles:
            straddled_slots.update(stra.get('slot_ids', []))

        for slot in self.slot_defs:
            sid = slot['id']
            pts = np.array(slot['polygon'], np.int32).reshape((-1, 1, 2))
            is_occ = states.get(sid, False)
            d_info = dwell.get(sid, {})
            elapsed = d_info.get('elapsed_seconds', 0.0)

            if sid in straddled_slots:
                color = (210, 50, 210)
            elif not is_occ:
                color = (46, 204, 113)
            else:
                color = (60, 60, 220)

            cv2.fillPoly(overlay, [pts], color)
            cv2.polylines(vis, [pts], True, color, 2, cv2.LINE_AA)

            cx = int(np.mean([p[0] for p in slot['polygon']]))
            cy = int(np.mean([p[1] for p in slot['polygon']]))

            if sid in straddled_slots:
                cv2.putText(vis, f"{slot['name']} : BLOQUE", (cx - 45, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 2)
            elif not is_occ:
                cv2.putText(vis, f"{slot['name']} : LIBRE", (cx - 40, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
            else:
                cid = d_info.get('car_id', '?')
                mins, secs = divmod(int(elapsed), 60)
                cv2.putText(vis, f"{slot['name']} [Car #{cid}]", (cx - 48, cy - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1)
                cv2.putText(vis, f"{mins:02d}m {secs:02d}s", (cx - 20, cy + 12), cv2.FONT_HERSHEY_SIMPLEX, 0.40, (0, 255, 255), 1)

        cv2.addWeighted(overlay, 0.28, vis, 0.72, 0, vis)

        # Véhicules
        for v in vehicles:
            x1, y1, x2, y2 = map(int, v['bbox'])
            px, py = int((x1 + x2) / 2), y2
            cv2.rectangle(vis, (x1, y1), (x2, y2), (220, 200, 100), 1)
            cv2.circle(vis, (px, py), 3, (0, 255, 255), -1)
            cv2.putText(vis, f"Car #{v['id']}", (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.38, (230, 230, 230), 1)

        # Alertes chevauchement
        for stra in straddles[:2]:
            x1, y1, x2, y2 = map(int, stra['bbox'])
            cv2.rectangle(vis, (x1, y1), (x2, y2), (210, 50, 210), 2)
            cv2.putText(vis, f"A CHEVAL SUR LA LIGNE (Car #{stra['car_id']})", (x1, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (210, 50, 210), 2)

        # Alertes collision
        for col in collisions[:2]:
            pt = col['pos']
            c1, c2 = col['car1_id'], col['car2_id']
            if col['type'] == 'COLLISION':
                cv2.circle(vis, pt, 28, (0, 0, 255), 2)
                cv2.circle(vis, pt, 14, (0, 80, 255), -1)
                cv2.putText(vis, f"IMPACT ! Car #{c1} x #{c2}", (pt[0] - 80, pt[1] - 35), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (0, 0, 255), 2)
            elif col['type'] == 'NEAR_MISS':
                cv2.circle(vis, pt, 24, (0, 165, 255), 2)
                cv2.putText(vis, f"NEAR-MISS ! Car #{c1}/#{c2}", (pt[0] - 75, pt[1] - 30), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 165, 255), 2)

        # HUD
        cv2.rectangle(vis, (0, 0), (w, 48), (18, 18, 18), -1)
        tot, free, occ = analytics['total_slots'], analytics['free_count'], analytics['occupied_count']
        rate = analytics['occupancy_rate']
        hud = f"SMART PARKING | Libres: {free}/{tot} | Occupees: {occ} ({rate:.0f}%) | Entrees: {entries} | FPS: {fps:.1f}"
        cv2.putText(vis, hud, (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (240, 240, 240), 1)
        return vis

def play_video(file_path):
    os.system(f"ffmpeg -y -i {file_path} -vcodec libx264 -crf 23 -pix_fmt yuv420p temp_play.mp4 > /dev/null 2>&1")
    target = 'temp_play.mp4' if os.path.exists('temp_play.mp4') else file_path
    mp4_data = open(target, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_data).decode()
    return HTML(f'''
    <div style="text-align: center; background-color: #1a1a1a; padding: 15px; border-radius: 8px;">
      <video width="800" controls autoplay loop style="border-radius: 6px;">
            <source src="{data_url}" type="video/mp4">
      </video>
    </div>
    ''')

print('[OK] Visualiseur & Lecteur vidéo initialisés !')


## 4. Expérience 1 : Stationnement Régulier & Conforme (Dwell Time)


In [ ]:
engine1 = SpatialEngine(PARKING_CONFIG)
vis1 = ParkingVisualizer(PARKING_CONFIG)
out1 = cv2.VideoWriter('exp1_proper_parking.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (1280, 720))

car_101 = {'id': 101, 'x': 50.0, 'y': 360.0}
car_102 = {'id': 102, 'x': 350.0, 'y': 190.0}
start_t = time.time()

for f_idx in range(240):
    sim_t = start_t + (f_idx / 30.0)
    frame = np.full((720, 1280, 3), (45, 45, 45), dtype=np.uint8)
    for x in range(0, 1280, 60):
        cv2.line(frame, (x, 360), (x + 30, 360), (0, 255, 255), 2)

    if f_idx < 40:
        car_101['x'] += (190.0 - 50.0) / 40.0
        car_101['y'] = 360.0
    elif f_idx < 80:
        car_101['x'] = 190.0
        car_101['y'] -= (360.0 - 190.0) / 40.0
    else:
        car_101['x'] = 190.0
        car_101['y'] = 190.0

    tracked = []
    for c in [car_101, car_102]:
        cx, cy = int(c['x']), int(c['y'])
        bbox = (cx - 40, cy - 55, cx + 40, cy + 55)
        tracked.append({'id': c['id'], 'bbox': bbox, 'conf': 0.95, 'cls_name': 'car'})
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (80, 80, 200), -1)

    analytics = engine1.update(tracked, current_time=sim_t)
    out1.write(vis1.render(frame, analytics, tracked))

out1.release()
print('✅ Expérience 1 terminée !')
play_video('exp1_proper_parking.mp4')


## 5. Expérience 2 : Stationnement à Cheval sur la Ligne (Incivilité)


In [ ]:
engine2 = SpatialEngine(PARKING_CONFIG)
vis2 = ParkingVisualizer(PARKING_CONFIG)
out2 = cv2.VideoWriter('exp2_line_straddling.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (1280, 720))

car_104 = {'id': 104, 'x': 60.0, 'y': 360.0}
start_t = time.time()

for f_idx in range(240):
    sim_t = start_t + (f_idx / 30.0)
    frame = np.full((720, 1280, 3), (45, 45, 45), dtype=np.uint8)
    for x in range(0, 1280, 60):
        cv2.line(frame, (x, 360), (x + 30, 360), (0, 255, 255), 2)

    if f_idx < 45:
        car_104['x'] += (270.0 - 60.0) / 45.0
        car_104['y'] = 360.0
    elif f_idx < 85:
        car_104['x'] = 270.0
        car_104['y'] += (530.0 - 360.0) / 40.0
    else:
        car_104['x'] = 270.0
        car_104['y'] = 530.0

    cx, cy = int(car_104['x']), int(car_104['y'])
    bbox = (cx - 40, cy - 55, cx + 40, cy + 55)
    tracked = [{'id': car_104['id'], 'bbox': bbox, 'conf': 0.95, 'cls_name': 'car'}]
    cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (80, 80, 200), -1)

    analytics = engine2.update(tracked, current_time=sim_t)
    out2.write(vis2.render(frame, analytics, tracked))

out2.release()
print('✅ Expérience 2 terminée !')
play_video('exp2_line_straddling.mp4')


## 6. Expérience 3 : Risque d'Accrochage Évité (Near-Miss)


In [ ]:
engine3 = SpatialEngine(PARKING_CONFIG)
vis3 = ParkingVisualizer(PARKING_CONFIG)
out3 = cv2.VideoWriter('exp3_near_miss.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (1280, 720))

car_101 = {'id': 101, 'x': 190.0, 'y': 190.0}
car_105 = {'id': 105, 'x': 50.0, 'y': 360.0}
start_t = time.time()

for f_idx in range(240):
    sim_t = start_t + (f_idx / 30.0)
    frame = np.full((720, 1280, 3), (45, 45, 45), dtype=np.uint8)
    for x in range(0, 1280, 60):
        cv2.line(frame, (x, 360), (x + 30, 360), (0, 255, 255), 2)

    car_105['x'] += 4.5
    if f_idx >= 15 and f_idx < 45:
        car_101['y'] += 3.5
    elif f_idx >= 45:
        car_101['y'] = 295.0 # Freinage in extremis

    tracked = []
    for c in [car_101, car_105]:
        cx, cy = int(c['x']), int(c['y'])
        bbox = (cx - 40, cy - 55, cx + 40, cy + 55)
        tracked.append({'id': c['id'], 'bbox': bbox, 'conf': 0.95, 'cls_name': 'car'})
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), (80, 80, 200), -1)

    analytics = engine3.update(tracked, current_time=sim_t)
    out3.write(vis3.render(frame, analytics, tracked))

out3.release()
print('✅ Expérience 3 (Near-Miss) terminée !')
play_video('exp3_near_miss.mp4')


## 7. Expérience 4 : Collision Réelle Confirmée (Crash en Allée)


In [ ]:
engine4 = SpatialEngine(PARKING_CONFIG)
vis4 = ParkingVisualizer(PARKING_CONFIG)
out4 = cv2.VideoWriter('exp4_actual_collision.mp4', cv2.VideoWriter_fourcc(*'mp4v'), 30.0, (1280, 720))

car_101 = {'id': 101, 'x': 190.0, 'y': 190.0}
car_105 = {'id': 105, 'x': 50.0, 'y': 360.0}
start_t = time.time()
collision_occurred = False

for f_idx in range(240):
    sim_t = start_t + (f_idx / 30.0)
    frame = np.full((720, 1280, 3), (45, 45, 45), dtype=np.uint8)
    for x in range(0, 1280, 60):
        cv2.line(frame, (x, 360), (x + 30, 360), (0, 255, 255), 2)

    if not collision_occurred:
        car_105['x'] += 4.0
        if f_idx >= 15 and car_101['y'] < 360.0:
            car_101['y'] += 4.0
        if abs(car_105['x'] - car_101['x']) < 70 and abs(car_105['y'] - car_101['y']) < 50:
            collision_occurred = True

    tracked = []
    for c in [car_101, car_105]:
        cx, cy = int(c['x']), int(c['y'])
        bbox = (cx - 40, cy - 55, cx + 40, cy + 55)
        tracked.append({'id': c['id'], 'bbox': bbox, 'conf': 0.95, 'cls_name': 'car'})
        color = (80, 80, 200) if not collision_occurred else (40, 40, 220)
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), color, -1)

    analytics = engine4.update(tracked, current_time=sim_t)
    vis_frame = vis4.render(frame, analytics, tracked)

    if collision_occurred:
        pulse = int((f_idx % 10) * 4)
        impact_pt = (int((car_101['x'] + car_105['x']) / 2), 360)
        cv2.circle(vis_frame, impact_pt, 40 + pulse, (0, 0, 255), 3, cv2.LINE_AA)

    out4.write(vis_frame)

out4.release()
print('✅ Expérience 4 (Collision Réelle) terminée !')
play_video('exp4_actual_collision.mp4')
